# V1DD tutorial

Objective: Explore and become familiar with the dataset

Learning goals:
- Use PyNWB to explore file structure

Deliverables:
- V1DD_README.md template
- Basic summary statistic figure
- Basic "processed data" figure

<span style="color:red">
**Teaching note:**
1. Walk students through navigating to this notebook. Open this notebook
2. Talk through the Objectives, Learning Goals, the structure (walk through tutorial together), and discuss the deliverables
3. Create the `code/V1DD_README.md` file, drag to the side, and walk through the template (very high-level scaffold) -- type in realtime. Explain that they will be filling it out side-by-side.
</span>

Scaffold (from word doc):
1. Data needs / expectations 

Does this dataset have everything I need to answer my question? Walk through each category: 

Neural activity — how is it packaged, raw or processed? 

V1DD: e.g. raw calcium TIFF movies (extract the signal yourself) vs. dF/F per ROI. 

Dynamic Routing: raw ephys file or spikes? 

Stimulus information — stimulus tables, how are tables organized, key vocabulary?  

Behavior — behavior tables (e.g. rewards: hits/misses) and behavior timeseries (running, pupil). 

Metadata — animal and genotype information, etc…  

2. Explore data structure and vocabulary 

All physiology datasets are in NWB format, accessible with PyNWB. Add short explanation of NWB.  

Code — using PyNWB, open an NWB file, show its structure, explore a bit. 

Important data to highlight:  

Neural activity  

Epochs/stimulus/behavior tables  

Behavior timeseries data (if available)  

Questions to ask:  

Does data fit my needs?  

Is anything missing?  

Any processing required before I can work with it?  

3. Extract the data pieces you need 

Pull out the neural activity first relative to stimulus or epoch. You'll likely want to shape the data so neural activity is aligned to different conditions — e.g. auditory vs. visual stimuli. 

Options for data to unpack and visualize:  

Neural activity,  plot a single unit or ROI. 

Plot the stimulus/trials/epochs tables. Unpack columns/keywords.  

Plot single unit neural activity aligned with stimulus/behavior  

QC – extract good quality units.  

Plot PSTH or averaged responses aligned to stimulus 

Maybe show what averages would look like if you include bad units  

4. Summary plots 

Show useful summary plots. Some options:  

Cross-session summary — # sessions/genotype  

Recording/session overview — session duration, number of trials, epochs  

Stimulus inventory — counts and durations per trial/stimulus type  

Behavior overview — hit/miss/false-alarm counts, performance over the session, running and pupil traces aligned to neural activity (?)  

Time alignment check — overlay stimulus onsets, licks/rewards, and neural activity on a shared clock for a few trials to confirm alignment. 

Population activity raster / heatmap — units or ROIs × time aligned to relevant stimuli  

Trial-averaged response (PSTH / mean dF/F)   

Responsiveness / selectivity — fraction of units responsive to the stimulus, and a simple selectivity or preference index across the population. 

Anatomical breakdown — counts per brain region  

ROI footprints / field-of-view — max-projection image with ROI masks overlaid  

In [1]:
# general imports 
import os
from pathlib import Path
import numpy as np
import pandas as pd 
import pynwb

import matplotlib.pyplot as plt
%matplotlib inline 

# nwb specific imports (from 2025)
# import pynwb
# from nwbwidgets import nwb2widget
 

!pip install -U hdmf-zarr==0.12.0

from hdmf_zarr import NWBZarrIO

In [2]:
# set data path
import sys
import platform
from os.path import join as pjoin

platstring = platform.platform()
system = platform.system()
if system == "Darwin":
    # macOS
    data_dir = "/Volumes/Brain2025/"
elif system == "Windows":
    # Windows (replace with the drive letter of USB drive)
    data_dir = "E:/"
elif "amzn" in platstring:  # Code Ocean
    data_dir = "/data/"
else:
    # then your own linux platform
    # EDIT location where you mounted hard drive
    data_dir = "/media/$USERNAME/Brain2025/"
    
print('data directory set to', data_dir)

data directory set to /data/


## Inspect metadata

<span style="color:red">**NB: Currently using `V1DD_metadata.csv` from 2025 SWDB workshop.**</span>

We should be update this to reflect the data that we are using this year! (what updates do we expect?!?)

In [3]:
# Load metadata CSV 
metadata = pd.read_csv('/data/metadata/V1DD_metadata.csv', index_col = False) 
metadata.head()  

,project_name,_id,name,subject_id,golden_mouse,genotype,date_of_birth,sex,modality,session_date,age,session_time,column,volume
0,V1 Deep Dive,3b85c659-20c8-438f-ab58-de1aac3b81cf,416296_2018-11-29_12-08-40_nwb_2025-08-08_16-2...,416296,False,Camk2a-tTA/wt;tetO-GCaMP6s/wt,2018-08-06,Female,"['Planar optical physiology', 'Behavior videos']",2018-11-29,115,12:08:40.014190,2,5
1,V1 Deep Dive,b98ce4a9-a66b-4b70-baa6-95ef451ae087,427836_2019-04-25_12-16-58_nwb_2025-08-08_16-2...,427836,False,Slc17a7-IRES2-Cre/wt;Camk2a-tTA/wt;Ai94(TITL-G...,2018-10-08,Female,"['Planar optical physiology', 'Behavior videos']",2019-04-25,199,12:16:58.240890,5,3
2,V1 Deep Dive,5cccd09c-4ae8-4d8e-ae92-23099d22bbe2,427836_2019-04-24_13-06-45_nwb_2025-08-08_16-2...,427836,False,Slc17a7-IRES2-Cre/wt;Camk2a-tTA/wt;Ai94(TITL-G...,2018-10-08,Female,"['Planar optical physiology', 'Behavior videos']",2019-04-24,198,13:06:45.257460,4,3
3,V1 Deep Dive,911d215f-dafa-4b39-86f7-b36336974da1,427836_2019-04-25_13-49-39_nwb_2025-08-08_16-2...,427836,False,Slc17a7-IRES2-Cre/wt;Camk2a-tTA/wt;Ai94(TITL-G...,2018-10-08,Female,"['Planar optical physiology', 'Behavior videos']",2019-04-25,199,13:49:39.163550,5,4
4,V1 Deep Dive,c918f874-d534-4341-b0dc-f1aba388af66,427836_2019-04-26_12-54-40_nwb_2025-08-08_16-2...,427836,False,Slc17a7-IRES2-Cre/wt;Camk2a-tTA/wt;Ai94(TITL-G...,2018-10-08,Female,"['Planar optical physiology', 'Behavior videos']",2019-04-26,200,12:54:40.332660,2,4


In [4]:
print("Unique subject ids:")
metadata.subject_id.unique()


Unique subject ids:


array([416296, 427836, 438833, 409828])

In [5]:
selected_path = \
    '/root/capsule/data/409828_V1DD_Filtered/409828_2018-11-06_14-02-59_filtered_2026-04-09_04-59-00/409828_2018-11-06_14-02-59.nwb.zarr'

# nwb = pynwb.read_nwb(selected_path)
# selected_path = metadata.get_df()["nwb_path"][3]

print(selected_path)
nwb = pynwb.read_nwb(selected_path)

/root/capsule/data/409828_V1DD_Filtered/409828_2018-11-06_14-02-59_filtered_2026-04-09_04-59-00/409828_2018-11-06_14-02-59.nwb.zarr


**Exercise: Document the data file structure**

```
- data/
  - <subject_id>_V1DD_Filtered /
    - <subject_id>_<session_date>_<some date>_filtered_<some_other_date> /
      - *.nwb.zarr  # this is the main data
      - original_metadata/  # original metadata, don't need to look
      - acquisition.json
      - # etc
```